# Multi-UAV Swarm Trajectory Planning

This notebook demonstrates centralized offline planning for a fleet of UAVs
navigating through a shared forest environment.  The pipeline:

1. Load the shared forest environment
2. Define a fleet of UAVs with start/goal pairs that share a narrow passage
3. Run `SwarmPlanner.plan_all()` — plans each UAV independently, detects
   inter-UAV conflicts, and resolves them via priority-based time shifting
4. Visualise results: static Matplotlib plots + interactive Three.js simulation

In [1]:
import sys, os

# Make sure the parent directory (uavsafeplanning/) is on the path
_swarm_dir  = os.path.abspath("")
_parent_dir = os.path.dirname(_swarm_dir)
_sto_dir    = os.path.join(_parent_dir, "STO")
for _p in [_swarm_dir, _parent_dir, _sto_dir]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

os.chdir(_parent_dir)

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Load Environment

In [2]:
from swarm import Environment

env = Environment(
    forest_path="traj_gen_utils/data/forest.txt",
    padding=0,
    astar_connectivity=18,
)
print(f"Grid shape : {env.vg.info.shape}")
print(f"Origin     : {env.vg.info.origin}")
print(f"Obstacles  : {env.obs_np.shape[0]} voxels")

Grid shape : (100, 100, 40)
Origin     : (0, 0, 0)
Obstacles  : 62560 voxels


## 2. Define UAV Fleet

Three UAVs whose A\* paths converge through the same narrow corridor region
around `(60, 60, 10)`.  This forces the conflict-detection & resolution
logic to activate.

In [3]:
from swarm import UAV

uavs = [
    UAV(uav_id="Alpha",   start=np.array([1, 5, 10]),   goal=np.array([90, 90, 10]), radius=1.0),
    UAV(uav_id="Bravo",   start=np.array([1, 10, 10]),  goal=np.array([90, 85, 10]), radius=1.0),
    UAV(uav_id="Charlie", start=np.array([5, 1, 10]),   goal=np.array([88, 92, 10]), radius=1.0),
]

print("Fleet:")
for u in uavs:
    print(f"  {u.uav_id}: {u.start} -> {u.goal}  (r={u.radius})")

Fleet:
  Alpha: [ 1  5 10] -> [90 90 10]  (r=1.0)
  Bravo: [ 1 10 10] -> [90 85 10]  (r=1.0)
  Charlie: [ 5  1 10] -> [88 92 10]  (r=1.0)


## 3. Run Swarm Planner

In [ ]:
from swarm import SwarmPlanner

sto_params = dict(
    v_max=2.0,
    a_max=1.0,
    lambda_jerk=10.0,
    lambda_time=0.05,
    lambda_vel=120.0,
    lambda_acc=5.0,
    lambda_corridor=1000.0,
    corridor_cost_type="l2",
    max_iter=50,
    adaptive_weights=True,
    weight_phases=3,
    weight_scale_factor=5.0,
)

sp = SwarmPlanner(
    env=env,
    uavs=uavs,
    sto_params=sto_params,
    conflict_dt=0.1,
    safety_margin=0.5,
    verbose=True,
)

results = sp.plan_all()

SWARM PLANNER
Fleet size: 3 UAVs
Safety margin: 0.5 m

[plan] UAV Alpha: [ 1  5 10] -> [90 90 10]


/home/okanarif/repositories/CopySemesterThesis/uavsafeplanning/STO/sto_planner.py:146: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  self.waypoints = torch.tensor([waypoints_init], dtype=torch.float32, requires_grad=True)


       done in 9.88s  (T=84.22s, segments=5)
[plan] UAV Bravo: [ 1 10 10] -> [90 85 10]


## 4. Results Summary

In [ ]:
print("=" * 60)
print("SWARM RESULTS")
print("=" * 60)
print(f"Total planning time:    {results['planning_time']:.2f} s")
print(f"Initial conflicts:      {len(results['conflicts_initial'])}")
print(f"Remaining conflicts:    {len(results['conflicts_remaining'])}")
print()

print(f"{'UAV':<10} {'Offset (s)':>10} {'Traj T (s)':>10} {'End (s)':>10} {'Plan (s)':>10} {'Segments':>8}")
print("-" * 60)
for u in uavs:
    pt = results['per_uav_times'].get(u.uav_id, 0)
    ns = len(u.A_list) if u.A_list else 0
    print(f"{u.uav_id:<10} {u.time_offset:>10.2f} {u.total_time():>10.2f} "
          f"{u.time_offset + u.total_time():>10.2f} {pt:>10.2f} {ns:>8}")
print("=" * 60)

## 5. Static Visualizations

In [ ]:
from swarm.visualizer_static import (
    plot_trajectories_3d,
    plot_min_distances,
    plot_velocity_profiles,
)

plot_trajectories_3d(uavs, title="Multi-UAV Trajectories")

dist_times, pair_dists = results["min_distance_profile"]
plot_min_distances(uavs, dist_times, pair_dists, safety_margin=0.5)

plot_velocity_profiles(uavs, v_max=2.0, a_max=1.0)

## 6. Interactive 3D Web Simulation

Generates a standalone HTML file with Three.js animation.
Open the file in a browser, or view it inline via `IFrame`.

In [ ]:
from swarm.visualizer_web import generate_simulation_html

html_path = generate_simulation_html(
    uavs,
    output_path="swarm/simulation.html",
    obs_np=env.obs_np,
    dt=0.05,
    safety_margin=0.5,
)
print(f"Simulation written to: {html_path}")

In [ ]:
from IPython.display import IFrame
IFrame(src="simulation.html", width=900, height=600)

## 7. Per-UAV Constraint Violations

In [ ]:
print(f"{'UAV':<10} {'Max Vel Viol':>14} {'Max Acc Viol':>14} {'Max Corr Viol':>14}")
print("-" * 56)
for u in uavs:
    v = u.sto_results['violations']
    print(f"{u.uav_id:<10} {v['vel']:>14.6f} {v['acc']:>14.6f} {v['corridor']:>14.6f}")